In [2]:
import weaviate
from weaviate.classes.config  import Configure
import requests, json

client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)

#Only need to run once to test create the collection

collections = client.collections.delete("Grounded_nomic")
print(collections)


questions = client.collections.create(
    name="Grounded_nomic",
    vector_config=Configure.Vectors.text2vec_ollama(  # Configure the Ollama embedding integration
        api_endpoint="http://172.17.0.6:11434",  # If using Docker you might need: http://host.docker.internal:11434
        model="nomic-embed-text:latest",  # The model to use
    ),
    generative_config=Configure.Generative.ollama(  # Configure the Ollama generative integration
        api_endpoint="http://172.17.0.6:11434",  # If using Docker you might need: http://host.docker.internal:11434
        model="llama3.2",  # The model to use
    ),
)


client.close()

None


In [3]:
client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)

with open("/home/prime/Documents/multimodal-oran-rag/Output/O-RAN.WG3.TS.E2AP-R004-v07.00/O-RAN.WG3.TS.E2AP-R004-v07.00_cleaned.json", 'r') as f:
    data = json.load(f)

vectorDB = client.collections.use("Grounded_nomic")

with vectorDB.batch.fixed_size(batch_size=200) as batch:
    for d in data:
        batch.add_object(
            {
                "page": d["page"],
                "Description": d["Description"],
                "Text": d["Text"],
                "Trace": d["Trace"],
            }
        )
        if batch.number_errors > 10:
            print("Batch import stopped due to excessive errors.")
            break

failed_objects = vectorDB.batch.failed_objects
if failed_objects:
    print(f"Number of failed imports: {len(failed_objects)}")
    print(f"First failed object: {failed_objects[0]}")

client.close()  # Free up resources

In [25]:
import weaviate
import json

client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)
questions = client.collections.use("Grounded_nomic")

response = questions.query.near_text(
    query="purpose of the E2 Connection Update procedure",
    limit=3
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=2))

client.close()  # Free up resources

{
  "text": "The purpose of the E2 Connection Update procedure is to allow the Near-RT RIC to update the TNL information associated with the E2 interface connection between the E2 Node and Near-RT RIC.",
  "trace": "8.2 RIC Functional procedures --> 8.3.6 E2 Connection Update procedure --> 8.3.6.1 General",
  "page": 47.0,
  "description": ""
}
{
  "text": "Upon reception of a E2 CONNECTION UPDATE message, the E2 Node shall update as follows:",
  "trace": "8.2 RIC Functional procedures --> 8.3.6 E2 Connection Update procedure --> 8.3.6.2 Successful operation",
  "page": 47.0,
  "description": ""
}
{
  "text": "The purpose of the E2 Node Configuration Update procedure is to update application level E2 Node configuration data needed for E2 Node and Near-RT RIC to interoperate correctly over the E2 interface and to support E2 Node initiated TNL association removal.",
  "trace": "8.2 RIC Functional procedures --> 8.3.5 E2 Node Configuration Update procedure --> 8.3.5.1 General",
  "descrip

In [30]:
import weaviate

client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)

questions = client.collections.use("Grounded_nomic")

response = questions.generate.near_text(
    query="What is a E2 interface?",
    limit=3,
    grouped_task="Provided detailed answer to best of your ability, and cite source using trace."
)

print(response.generative.text)  # Inspect the generated text

client.close()  # Free up resources

Based on the provided text, here is a detailed explanation of the E2 interface and its protocol:

**What is the E2 interface?**

The E2 interface is a means for interconnecting a Near-RT RIC (Radio Resource Management Function) and an E2 Node (E-UTRA UEs). It provides a way for these two entities to communicate with each other.

**What is E2AP?**

E2AP (E2 Application Protocol) is a protocol that supports the functions of the E2 interface by signaling procedures defined in this document. It is developed in accordance with the general principles stated in O-RAN WG3 TS.E2GAP [2].

**Structure of E2AP procedures**

The E2AP procedures are divided into two modules:

* [5.1 E2AP procedure modules]

Unfortunately, the provided text does not provide further information on what these modules entail.

**E2 Node Component Interface Type**

An IE (Item) is used to identify an E2 node component type. Specifically, this IE is referred to in section 9.2.26 of the document, where it states:

"E2 Node